Bronze layer: read from landing and write into Delta

In [0]:
from pyspark.sql import functions as F

LANDING_PATH = "/Volumes/nyc_taxi/landing/raw_files/yellow_tripdata/"
BRONZE_TABLE = "nyc_taxi.bronze.yellow_trips"

In [0]:
bronze_df = (
    spark.read.parquet(LANDING_PATH)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

In [0]:
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE)
)

In [0]:
%sql
COMMENT ON TABLE nyc_taxi.bronze.yellow_trips IS
  'Raw NYC yellow taxi trips 2019-01..03 as delivered by TLC (parquet), plus ingestion metadata. No filtering or type changes.';

In [0]:
bronze = spark.table(BRONZE_TABLE)
print(f"bronze rows: {bronze.count():,}   (expected 22,612,607)")
display(bronze.groupBy("_source_file").count().orderBy("_source_file"))

In [0]:
%sql
DESCRIBE DETAIL nyc_taxi.bronze.yellow_trips;

In [0]:
%sql
DESCRIBE HISTORY nyc_taxi.bronze.yellow_trips;

In [0]:
ZONE_PATH  = "/Volumes/nyc_taxi/landing/raw_files/reference/taxi_zone_lookup.csv"
ZONE_TABLE = "nyc_taxi.bronze.taxi_zone_lookup"

(
    spark.read.option("header", True).option("inferSchema", True).csv(ZONE_PATH)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
    .write.format("delta").mode("overwrite").saveAsTable(ZONE_TABLE)
)

display(spark.table(ZONE_TABLE))